# 03 — Agent Traces Explorer

Run the agentic chatbot interactively, inspect each step, and review the
traces + logs it produces.

**Prerequisites:**
- LanceDB indexes built (vector + keyword FTS)
- Local model server running: `bash scripts/start_model_server.sh`
- (Optional) Phoenix running: `phoenix serve` for trace UI

## 0 — Parameters

In [ ]:
QUESTION     = "When did the Roman Republic end?"
SEARCH_TOP_K = 5     # chunks to retrieve
OPEN_TOP_N   = 3     # chunks to open and read
MAX_STEPS    = 10    # safety limit

## 1 — Run the chatbot

In [ ]:
from workbench.systems.chatbot import ChatbotSystem

system = ChatbotSystem()
state = system.ask(
    QUESTION,
    search_top_k=SEARCH_TOP_K,
    open_top_n=OPEN_TOP_N,
    max_steps=MAX_STEPS,
)

## 2 — Inspect the answer

In [ ]:
print(f"Question: {state.question}")
print(f"Steps taken: {state.step}")
print(f"Error: {state.error}")
print()
print("=" * 60)
print("ANSWER:")
print("=" * 60)
print(state.answer_text)
print()
print(f"Citations: {state.citations}")
print(f"Tokens in/out: {state.tokens_in} / {state.tokens_out}")
print(f"Model latency: {state.model_latency_ms:.0f} ms")

## 3 — Inspect retrieved chunks (before opening)

In [ ]:
import pandas as pd

if state.retrieved:
    rows = [
        {
            "rank": i + 1,
            "chunk_id": r.chunk_id,
            "title": r.title,
            "score": f"{r.score:.4f}",
            "snippet": r.snippet[:80] + "…",
        }
        for i, r in enumerate(state.retrieved)
    ]
    df_retrieved = pd.DataFrame(rows)
    display(df_retrieved)
else:
    print("No chunks retrieved.")

## 4 — Inspect opened chunks (full evidence)

In [ ]:
for i, chunk in enumerate(state.opened, 1):
    cited = "✅ CITED" if chunk.chunk_id in state.citations else ""
    print(f"\n{'=' * 60}")
    print(f"Evidence {i}: [{chunk.chunk_id}] {chunk.title} {cited}")
    if chunk.section:
        print(f"Section: {chunk.section}")
    print(f"{'=' * 60}")
    print(chunk.text[:500])
    if len(chunk.text) > 500:
        print(f"... ({len(chunk.text)} chars total)")

## 5 — Read the run log

In [ ]:
import json
from pathlib import Path

# Find the most recent log file
log_dir = Path("../runs/logs")
log_files = sorted(log_dir.glob("*.jsonl"), reverse=True)

if log_files:
    latest = log_files[0]
    print(f"Reading: {latest.name}\n")
    with open(latest) as f:
        for line in f:
            event = json.loads(line)
            ts = event.get("timestamp", "")[-12:]  # last 12 chars of ISO time
            etype = event.get("event_type", "")
            level = event.get("level", "")
            print(f"  {ts}  {level:<7}  {etype}")
else:
    print("No log files found. Is runs/logs/ empty?")

## 6 — Inspect the full prompt sent to the model

Reconstruct the prompt the agent built so you can see exactly what the LLM saw.

In [ ]:
from workbench.agents.loops import _SYSTEM_PROMPT, _build_evidence_block

evidence = _build_evidence_block(state.opened)

user_message = (
    f"Evidence:\n{evidence}\n\n"
    f"Question: {state.question}\n\n"
    "Answer the question using the evidence above. "
    "Cite sources using [chunk_id] notation."
)

print("=== SYSTEM ===")
print(_SYSTEM_PROMPT)
print()
print("=== USER ===")
print(user_message[:2000])
if len(user_message) > 2000:
    print(f"\n... ({len(user_message)} chars total)")

## 7 — Batch: try multiple questions

In [ ]:
questions = [
    "What caused the French Revolution?",
    "How does photosynthesis work?",
    "Who was Julius Caesar?",
]

results = []
for q in questions:
    print(f"\n--- Asking: {q} ---")
    s = system.ask(q, search_top_k=3, open_top_n=2)
    results.append({
        "question": q,
        "steps": s.step,
        "retrieved": len(s.retrieved),
        "opened": len(s.opened),
        "citations": len(s.citations),
        "answer_preview": s.answer_text[:100] + "…",
        "tokens_in": s.tokens_in,
        "tokens_out": s.tokens_out,
        "error": s.error,
    })

df_batch = pd.DataFrame(results)
display(df_batch)

## 8 — Metrics summary from SQLite

In [ ]:
import sqlite3

db_path = Path("../runs/metrics/metrics.sqlite")

if db_path.exists():
    conn = sqlite3.connect(str(db_path))

    print("=== Component latency (last 10) ===")
    df_lat = pd.read_sql_query(
        "SELECT run_id, component_name, duration_ms "
        "FROM component_latency ORDER BY id DESC LIMIT 10",
        conn,
    )
    display(df_lat)

    print("\n=== Model tokens (last 10) ===")
    df_tok = pd.read_sql_query(
        "SELECT run_id, model_name, tokens_in, tokens_out, "
        "       duration_ms, tokens_per_second "
        "FROM model_tokens ORDER BY id DESC LIMIT 10",
        conn,
    )
    display(df_tok)

    print("\n=== Retrieval counts (last 10) ===")
    df_ret = pd.read_sql_query(
        "SELECT run_id, query, keyword_candidates, vector_candidates, "
        "       reranked, final_count, duration_ms "
        "FROM retrieval_counts ORDER BY id DESC LIMIT 10",
        conn,
    )
    display(df_ret)

    conn.close()
else:
    print(f"Metrics DB not found at {db_path}")

---

### Things to try next

- Open **Phoenix** at http://localhost:6006 and find the traces for these runs
- Check that each trace shows: `agent.run` → `agent.step` (×3) → `tool.search_wikipedia` / `tool.open_chunk` / `model.generate`
- Change `OPEN_TOP_N` to 1 vs 5 and see how the answer quality changes
- Try questions that are NOT in Simple English Wikipedia — the agent should say it can't find enough evidence
- Look at the raw JSONL log file to understand what gets recorded per event